In [15]:
import requests
import pandas as pd
import numpy as np

In [22]:
url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 52.52,
    "longitude": 13.41,
    "hourly": (
        "temperature_2m,"
        "relative_humidity_2m,"
        "wind_speed_10m,"
        "wind_direction_10m,"
        "shortwave_radiation"
    ),
    "timezone": "Europe/Warsaw",
    "past_days": 1
}

response = requests.get(url, params=params)
data = response.json()

print("Status code:", response.status_code)

if response.status_code == 200:
    data = response.json()
    print("JSON received. Keys:", list(data.keys()))
else:
    print("Request failed.")

Status code: 200
JSON received. Keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']


In [23]:
df = pd.DataFrame({
    "time": data["hourly"]["time"],
    "temperature_2m": data["hourly"]["temperature_2m"],
    "relative_humidity_2m": data["hourly"]["relative_humidity_2m"],
    "wind_speed_10m": data["hourly"]["wind_speed_10m"],
    "wind_direction_10m": data["hourly"]["wind_direction_10m"],
    "shortwave_radiation": data["hourly"]["shortwave_radiation"]
})

df.head()

,time,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,shortwave_radiation
0,2025-11-23T00:00,-2.2,86,6.2,170,0.0
1,2025-11-23T01:00,-2.6,89,6.2,170,0.0
2,2025-11-23T02:00,-2.8,89,6.5,174,0.0
3,2025-11-23T03:00,-3.0,88,6.8,180,0.0
4,2025-11-23T04:00,-3.3,90,7.3,169,0.0


In [24]:
print(f'df.shape = {df.shape}')

df.shape = (192, 6)


In [25]:
df["time"] = pd.to_datetime(df["time"], utc=True)
df["time"].dtype

datetime64[ns, UTC]

In [26]:
df = df.set_index("time")
df.head()

,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,shortwave_radiation
time,,,,,
2025-11-23 00:00:00+00:00,-2.2,86,6.2,170,0.0
2025-11-23 01:00:00+00:00,-2.6,89,6.2,170,0.0
2025-11-23 02:00:00+00:00,-2.8,89,6.5,174,0.0
2025-11-23 03:00:00+00:00,-3.0,88,6.8,180,0.0
2025-11-23 04:00:00+00:00,-3.3,90,7.3,169,0.0


In [27]:
df = df.astype({
    "relative_humidity_2m": "float64",
    "wind_direction_10m": "float64"
})

In [28]:
df.dtypes

,0
temperature_2m,float64
relative_humidity_2m,float64
wind_speed_10m,float64
wind_direction_10m,float64
shortwave_radiation,float64


In [29]:
df.describe()

,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,shortwave_radiation
count,192.000000,192.000000,192.000000,192.000000,192.000000
mean,1.745312,86.057292,7.417708,161.911458,29.867708
std,2.478489,8.905075,2.529387,78.913416,56.701642
min,-3.800000,56.000000,1.300000,4.000000,0.000000
25%,0.600000,81.750000,5.975000,128.000000,0.000000
50%,2.000000,88.000000,7.350000,170.500000,0.000000
75%,3.525000,93.000000,8.825000,203.250000,29.625000
max,6.100000,99.000000,12.900000,356.000000,239.500000


In [30]:
df.isna().sum()

,0
temperature_2m,0
relative_humidity_2m,0
wind_speed_10m,0
wind_direction_10m,0
shortwave_radiation,0


In [33]:
def get_schema(df):
    return (df.columns.tolist(), df.dtypes.tolist())

schema1 = get_schema(df)

# simulate second run (or re-run your ETL code)
schema2 = get_schema(df)

print("Schemas identical:", schema1 == schema2)

Schemas identical: True


In [32]:
import os

file_path = "weather.parquet"
df.to_parquet(file_path)

if os.path.exists(file_path):
    print(f"File saved successfully: {file_path}")

File saved successfully: weather.parquet
